## Задание
Цель:
Применить на практике алгоритмы по автоматической оптимизации параметров моделей машинного обучения.


Этапы работы:

1. Получите данные и загрузите их в рабочую среду. (Jupyter Notebook или другую)
2. Подготовьте датасет к обучению моделей:
- Категориальные переменные переведите в цифровые значения. Можно использовать pd.get_dummies, preprocessing.LabelEncoder. Старайтесь не использовать для этой задачи циклы.
3. Разделите выборку на обучающее и тестовое подмножество. 80% данных оставить на обучающее множество, 20% на тестовое.
4. Обучите модель логистической регрессии с параметрами по умолчанию.
5. Подсчитайте основные метрики модели. Используйте следующие метрики и функцию:
cross_validate(…, cv=10, scoring=[‘accuracy’,‘recall’,‘precision’,‘f1’])
6. Оптимизируйте 3-4 параметра модели:
- Используйте GridSearchCV.
- Используйте RandomizedSearchCV.
- *Добавьте в п. 6b 2-5 моделей классификации и вариации их параметров.
- Повторите п. 5 после каждого итогового изменения параметров.
7. Сформулируйте выводы по проделанной работе:
a) Сравните метрики построенных моделей.
b) *Сравните с полученными результатами в домашнем задании по теме «Ансамблирование».

In [131]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import f1_score, accuracy_score

In [24]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
data_heart = pd.read_csv('/Users/sofagusina/Desktop/программирование/machine_learning/machine_learning/ML/Работа с признаками/Улучшение качества модели/heart 2.csv')
data_heart

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


In [5]:
data_heart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [7]:

label =LabelEncoder()
label.fit(data_heart['RestingECG'])

data_heart['RestingECG_encod'] = label.transform(data_heart['RestingECG'])

In [11]:

label.fit(data_heart['ExerciseAngina'])
data_heart['ExerciseAngina_encod'] = label.transform(data_heart['ExerciseAngina'])

In [10]:
label.fit(data_heart['ST_Slope'])
data_heart['ST_Slope_encod'] = label.transform(data_heart['ST_Slope'])

In [9]:
label.fit(data_heart['ChestPainType'])
data_heart['ChestPainType_encod'] = label.transform(data_heart['ChestPainType'])

In [8]:
label.fit(data_heart['Sex'])
data_heart['Sex_encod'] = label.transform(data_heart['Sex'])

In [12]:
data_heart


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,RestingECG_encod,Sex_encod,ChestPainType_encod,ST_Slope_encod,ExerciseAngina_encod
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0,1,1,1,2,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1,1,0,2,1,0
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0,2,1,1,2,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1,1,0,0,1,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0,1,1,2,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1,1,1,3,1,0
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1,1,1,0,1,0
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1,1,1,0,1,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1,0,0,1,1,0


Обучение модели

In [13]:
columns_drop = ['HeartDisease'] + list(data_heart.select_dtypes('object').columns)
X = data_heart.drop(columns=columns_drop)
y = data_heart['HeartDisease']

In [16]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [25]:
model_log = LogisticRegression()
model_log.fit(X_train,y_train)

LogisticRegression()

In [48]:
y_pred = model_log.predict(X_test)

In [49]:
from sklearn.metrics import  classification_report
print(classification_report(y_test.to_numpy(), y_pred.astype(int)))

              precision    recall  f1-score   support

           0       0.78      0.87      0.82        77
           1       0.90      0.82      0.86       107

    accuracy                           0.84       184
   macro avg       0.84      0.85      0.84       184
weighted avg       0.85      0.84      0.84       184



Расчет основных метрик

In [29]:
from sklearn.model_selection import cross_validate
scores = cross_validate(model_log, X_train, y_train, cv=10, scoring=['accuracy','recall','precision','f1'])
scores

{'fit_time': array([0.01614904, 0.00983191, 0.00940108, 0.01248217, 0.01020312,
        0.00931191, 0.00913787, 0.00905299, 0.00952601, 0.00916409]),
 'score_time': array([0.00411701, 0.00408292, 0.00393391, 0.00516081, 0.00352597,
        0.00361609, 0.00376391, 0.00374699, 0.00365615, 0.00349069]),
 'test_accuracy': array([0.89189189, 0.85135135, 0.89189189, 0.83783784, 0.79452055,
        0.84931507, 0.83561644, 0.89041096, 0.87671233, 0.80821918]),
 'test_recall': array([0.875     , 0.85      , 0.875     , 0.85365854, 0.85      ,
        0.875     , 0.875     , 0.9       , 0.9       , 0.9       ]),
 'test_precision': array([0.92105263, 0.87179487, 0.92105263, 0.85365854, 0.79069767,
        0.85365854, 0.83333333, 0.9       , 0.87804878, 0.7826087 ]),
 'test_f1': array([0.8974359 , 0.86075949, 0.8974359 , 0.85365854, 0.81927711,
        0.86419753, 0.85365854, 0.9       , 0.88888889, 0.8372093 ])}

In [132]:
print(f"accuracy - {accuracy_score(model_log.predict(X_test), y_test)}, f1-мера - {f1_score(model_log.predict(X_test), y_test)}")

accuracy - 0.842391304347826, f1-мера - 0.8585365853658536


В процессе обучения модели контролировалось, чтобы она не переобучалась, с помощью валидационных семплов. Оценивая основные метрики на валидационных семплах, можно сделать вывод, что модель показывает достаточно невысокие показатели precision, accuracy и f1 и высокий показатель recall. Проведем подбор гиперпараметров при помощи алгоритмов

Для модели логистической регрессии в качестве гиперпараметров выберем параметр penalty (вид регуляризации), solver (тип решателя - проверим, влияет ли он на точность модели на данном датасете, поскольку классификация бинарная, дата-сет небольшой, то возможно влияние типа решателя будет несущественным), max_iter (максимальльное число итераций для сходимости решателя)

Подбор гиперпараметров с помощью модели GridSearch

Так как каждый тип решатель поддерживает только конкретный тип регуляризации, надо учесть это при составлении словаря гиперпараметров

In [101]:
from sklearn.model_selection import GridSearchCV
param_grid = [{
    'max_iter':[50, 100, 200, 500,2000,5000],
    'penalty':['l1','l2'],
    'solver':['linear']
},
{
    'max_iter':[50, 100, 200, 500,2000,5000],
    'penalty':['l2','None'],
    'solver':['sag']
},
{
    'max_iter':[50, 100, 200, 500,2000,5000],
    'penalty':['l2','None','elasticnet','l1'],
    'solver':['saga']
},
 ] ## 'elasticnet' - l1 и l2
print(param_grid)

[{'max_iter': [50, 100, 200, 500, 2000, 5000], 'penalty': ['l1', 'l2'], 'solver': ['linear']}, {'max_iter': [50, 100, 200, 500, 2000, 5000], 'penalty': ['l2', 'None'], 'solver': ['sag']}, {'max_iter': [50, 100, 200, 500, 2000, 5000], 'penalty': ['l2', 'None', 'elasticnet', 'l1'], 'solver': ['saga']}]


In [87]:

grid = GridSearchCV(model_log, param_grid, cv=10, scoring=['accuracy','recall','precision','f1'], n_jobs=-1, verbose=1,refit='accuracy')
grid.fit(X_train, y_train)

Fitting 10 folds for each of 48 candidates, totalling 480 fits


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max

GridSearchCV(cv=10, estimator=LogisticRegression(), n_jobs=-1,
             param_grid=[{'max_iter': [50, 100, 200, 500, 2000, 5000],
                          'penalty': ['l1', 'l2'], 'solver': ['linear']},
                         {'max_iter': [50, 100, 200, 500, 2000, 5000],
                          'penalty': ['l2', 'None'], 'solver': ['sag']},
                         {'max_iter': [50, 100, 200, 500, 2000, 5000],
                          'penalty': ['l2', 'None', 'elasticnet', 'l1'],
                          'solver': ['saga']}],
             refit='accuracy',
             scoring=['accuracy', 'recall', 'precision', 'f1'], verbose=1)

In [93]:
grid.best_estimator_

LogisticRegression(max_iter=5000, solver='saga')

In [94]:
grid.best_estimator_.score(X_test, y_test)

0.8315217391304348

In [130]:

print(f"accuracy - {accuracy_score(grid.best_estimator_.predict(X_test), y_test)}, f1-мера - {f1_score(grid.best_estimator_.predict(X_test), y_test)}")



accuracy - 0.8315217391304348, f1-мера - 0.8472906403940886


solver='saga' — стохастический градиентный спуск, эффективен для больших выборок;

max_iter=5000 — лучшее число итераций, обеспечившее сходимость;

penalty='l2' — по умолчанию для solver='saga', поэтому не отображается в строке — но всё равно используется

Проверим эффективность выбранной модели

In [126]:
model_new_log = LogisticRegression(max_iter=5000, solver='saga')
model_new_log.fit(X_train,y_train)

LogisticRegression(max_iter=5000, solver='saga')

In [127]:
model_new_log.score(X_test,y_test)

0.8315217391304348

In [128]:
scores_new = cross_validate(model_new_log, X_train, y_train, cv=10, scoring=['accuracy','recall','precision','f1'])
scores_new

{'fit_time': array([0.41377878, 0.40220284, 0.4524138 , 0.40188909, 0.37039495,
        0.40274405, 0.40337205, 0.40316796, 0.403198  , 0.40290785]),
 'score_time': array([0.00531888, 0.00394917, 0.00424027, 0.00384092, 0.00396299,
        0.00398898, 0.00396419, 0.00405884, 0.00398707, 0.00692511]),
 'test_accuracy': array([0.89189189, 0.86486486, 0.89189189, 0.81081081, 0.83561644,
        0.83561644, 0.84931507, 0.8630137 , 0.87671233, 0.79452055]),
 'test_recall': array([0.875     , 0.85      , 0.875     , 0.82926829, 0.85      ,
        0.825     , 0.875     , 0.85      , 0.925     , 0.9       ]),
 'test_precision': array([0.92105263, 0.89473684, 0.92105263, 0.82926829, 0.85      ,
        0.86842105, 0.85365854, 0.89473684, 0.86046512, 0.76595745]),
 'test_f1': array([0.8974359 , 0.87179487, 0.8974359 , 0.82926829, 0.85      ,
        0.84615385, 0.86419753, 0.87179487, 0.89156627, 0.82758621])}

Все метрики после подбора гиперпараметров уменьшились, значит, гиперпараметры, установленные по умолчанию в алгоритме, лучше

Подбор параметров RandomizedSearchCV

In [102]:
from sklearn.model_selection import RandomizedSearchCV

grid_random= RandomizedSearchCV(model_log, param_grid, n_iter=10,
                          cv=10, scoring='accuracy', verbose=1)

grid.fit(X_train,y_train)

Fitting 10 folds for each of 48 candidates, totalling 480 fits


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max

GridSearchCV(cv=10, estimator=LogisticRegression(), n_jobs=-1,
             param_grid=[{'max_iter': [50, 100, 200, 500, 2000, 5000],
                          'penalty': ['l1', 'l2'], 'solver': ['linear']},
                         {'max_iter': [50, 100, 200, 500, 2000, 5000],
                          'penalty': ['l2', 'None'], 'solver': ['sag']},
                         {'max_iter': [50, 100, 200, 500, 2000, 5000],
                          'penalty': ['l2', 'None', 'elasticnet', 'l1'],
                          'solver': ['saga']}],
             refit='accuracy',
             scoring=['accuracy', 'recall', 'precision', 'f1'], verbose=1)

In [106]:

grid.best_estimator_

LogisticRegression(max_iter=5000, solver='saga')

In [109]:
grid.best_estimator_.score(X_test, y_test)

0.8315217391304348

In [111]:
scores_new_random = cross_validate(grid.best_estimator_, X_train, y_train, cv=10, scoring=['accuracy','recall','precision','f1'])
scores_new_random

{'fit_time': array([0.40954709, 0.40636873, 0.40308404, 0.40284109, 0.37027383,
        0.40272784, 0.40286589, 0.421242  , 0.4057529 , 0.46506429]),
 'score_time': array([0.00423503, 0.00422502, 0.00405908, 0.00389075, 0.00378013,
        0.00397706, 0.00402403, 0.00395298, 0.00398803, 0.00493073]),
 'test_accuracy': array([0.89189189, 0.86486486, 0.89189189, 0.81081081, 0.83561644,
        0.83561644, 0.84931507, 0.8630137 , 0.87671233, 0.79452055]),
 'test_recall': array([0.875     , 0.85      , 0.875     , 0.82926829, 0.85      ,
        0.825     , 0.875     , 0.85      , 0.925     , 0.9       ]),
 'test_precision': array([0.92105263, 0.89473684, 0.92105263, 0.82926829, 0.85      ,
        0.86842105, 0.85365854, 0.89473684, 0.86046512, 0.76595745]),
 'test_f1': array([0.8974359 , 0.87179487, 0.8974359 , 0.82926829, 0.85      ,
        0.84615385, 0.86419753, 0.87179487, 0.89156627, 0.82758621])}

Модель GridSearch и RandomSearchCV выбрали в качестве лучшей модели - модель с одинаковыми гиперпараметрами.

Проверим разные модели с разными гиперпараметрами для задачи классификации 

In [114]:
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

In [122]:
models=[ 
      {'name':'RF',"model": RandomForestClassifier(), 'params':{'n_estimators':[10,25,50,100,150,200], 'criterion':['gini', 'entropy'], 'max_depth':[3,5,7,9,11]}},
      {'name':'KN',"model": KNeighborsClassifier(), 'params':{'n_neighbors':list(range(1,30)),'weights': ['uniform', 'distance'], 'p':[1,2,3]}},
      {'name':'DT',"model": DecisionTreeClassifier(), 'params':{'criterion':['gini', 'entropy'], 'max_depth':[3,5,7,9,11]}}

]

res=[]
for v in  tqdm(models):
    tmp_model = RandomizedSearchCV(v['model'], v['params'], cv=5)
    tmp_model.fit(X_train, y_train)
    res.append(((v['name'], tmp_model, {'metrics':{'accuracy': accuracy_score(tmp_model.predict(X_test), y_test),
                                                 'f1-score':f1_score(tmp_model.predict(X_test), y_test)}})))

100%|██████████| 3/3 [00:06<00:00,  2.31s/it]


In [123]:
res

[('RF',
  RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(),
                     param_distributions={'criterion': ['gini', 'entropy'],
                                          'max_depth': [3, 5, 7, 9, 11],
                                          'n_estimators': [10, 25, 50, 100, 150,
                                                           200]}),
  {'metrics': {'accuracy': 0.8858695652173914,
    'f1-score': 0.9004739336492891}}),
 ('KN',
  RandomizedSearchCV(cv=5, estimator=KNeighborsClassifier(),
                     param_distributions={'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8,
                                                          9, 10, 11, 12, 13, 14,
                                                          15, 16, 17, 18, 19, 20,
                                                          21, 22, 23, 24, 25, 26,
                                                          27, 28, 29],
                                          'p': [1, 2, 3],
                     

In [124]:
for r in res:
    print(r[0], r[1].best_score_, r[1].best_params_)

RF 0.871922467617184 {'n_estimators': 100, 'max_depth': 11, 'criterion': 'entropy'}
KN 0.7520175193365017 {'weights': 'uniform', 'p': 1, 'n_neighbors': 11}
DT 0.8310595471065139 {'max_depth': 5, 'criterion': 'entropy'}


Самую высокую accuracy и f1-меру для данного дата-сета имеет модель случайного леса с количеством деревьев - 100, глубиной - 11, и информационном критерием "энтропия"

Вывод: работа состояла из нескольких частей
1) обучение модели логистической регрессии для классификации переменной HeartDisease на параметрах, выстроенных моделью по умолчанию (посмотреть параметры можно в документации алгоритма). Конечная accuracy данной модели - 84,2%, f1-мера - 85,8% 
2) подбор гиперпараметров модели через GridSearch и RandomSearch и обучение модели логистической регрессии на подобранных параметрах. Самой лучшей моделью оказалась модель с решателем типа saga (стохастический градиентный спуск) и числом итераций 5000.  Конечная accuracy данной модели - 83,1%, f1-мера - 84,7%. Метрики снизились, следовательно, параметры, установленные в алгоритме по умолчанию, лучше справляются с поставленной задачей. 
3) перебор трех разных моделей - метод ближайших соседей, случайный лес и деревья решений - и поиск наиболее эффективной модели с конкретными гиперпараметрами. Самой эффективной оказалась модель случайного леса с количеством деревьев - 100, глубиной - 11, и информационном критерием "энтропия".  Конечная accuracy данной модели - 88,5%, f1-мера - 90,4% 

Таким образом, модель с самыми высокими показателями - модель случайного леса, что логично, поскольку это ансамблевый алгоритм, который имеет априори выше эффективность работы
